# 02 · Movement labelling

| | |
|---|---|
| **入力** | `data/csv/<subject>/<session>.csv` |
| **出力** | `data/formatted/<subject>/<session>.csv`（`label` 列を追加） |

プロトコル上は 1 セッションにつき 1 動作を教示するが, 各フェーズの **開始・終了時刻**は信号から復元する必要がある. 2 つの手がかりを併用する.

* **姿勢** — 腰マーカー高さのしきい値で立位／座位を分ける.
* **動作開始** — EMG 活動包絡（帯域通過 → ヒルベルト → 低域通過）が安静時   ベースラインのしきい値を上回った時点.

生ラベルはその後クリーンアップする: 短い欠損を埋め, 1 サンプルだけの ちらつきを多数決フィルタで除去する.

7 クラス: `stand`, `sit`, `descending`, `ascending`, `walk`, `stand_up`, `sit_down`.

In [ ]:
import sys
from pathlib import Path

# notebooks/ から実行したときに motion_intent パッケージを import 可能にする
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from motion_intent import config

In [ ]:
from motion_intent.labeling import (
    emg_envelope, onset_from_envelope, threshold_state,
    fill_short_gaps, majority_smooth, label_transitions,
)

## Load a session

In [ ]:
csv_files = sorted(config.CSV_DIR.rglob('*.csv'))
assert csv_files, f'no session csv under {config.CSV_DIR} - run notebook 01 first'
print(f'{len(csv_files)} session csv files')

df = pd.read_csv(csv_files[0])

## Posture from hip height

`Markers_*FTC_*` は大転子（腰）の位置を追う. セッションごとに調整した 高さしきい値で立位と座位を分離する.

In [ ]:
HIP_COL = 'Markers_RFTC_Y'      # 鉛直軸
SIT_STAND_THRESHOLD = 0.75      # [m]; 撮影ボリュームごとに調整

if df is not None and HIP_COL in df:
    hip = df[HIP_COL].interpolate()
    posture = threshold_state(hip, SIT_STAND_THRESHOLD, 'stand', 'sit')
    df['posture'] = posture

## Movement onset from EMG

大腿四頭筋（`emg_RF_*`）のバーストが立ち上がり動作や歩行開始の起点になる. 検出した開始時刻を姿勢変化と突き合わせ, 遷移クラス（`stand_up`, `sit_down`）や 移動クラスを割り当てる.

In [ ]:
fs = config.FS
EMG_ONSET_CH = 'EMG_EMG0'       # リネームは 03 で実施; ここでは生の列名

if df is not None and EMG_ONSET_CH in df:
    env = emg_envelope(df[EMG_ONSET_CH].fillna(0).to_numpy(), fs=fs,
                       band=config.EMG_BAND)
    onsets = onset_from_envelope(env, fs=fs, k=3.0)
    print(f'{len(onsets)} EMG onsets')

## Assemble and clean the label track

以下のプレースホルダを, セッション固有のルール（姿勢区間を EMG 開始で分割, かかとマーカーからの段差接地判定 など）に置き換える. クリーンアップ処理は共通.

In [ ]:
if df is not None:
    raw_labels = pd.Series(df.get('posture'), index=df.index)   # プレースホルダ

    labels = fill_short_gaps(raw_labels, max_gap=int(0.3 * fs))
    labels = pd.Series(majority_smooth(labels.to_numpy(), win=int(0.2 * fs)),
                       index=df.index)
    df['label'] = labels

    print(df['label'].value_counts(dropna=False))
    print(f"{len(label_transitions(df['label'].to_numpy()))} transitions")

## Save

In [ ]:
for csv_path in csv_files:
    session = pd.read_csv(csv_path)
    # ... 上のラベル付けルールを session に適用する ...
    out_path = config.FORMATTED_DIR / csv_path.parent.name / csv_path.name
    out_path.parent.mkdir(parents=True, exist_ok=True)
    session.to_csv(out_path, index=False)
    print(out_path.relative_to(config.DATA_DIR))